# Week 4 — LSTM Baseline (MVP v0.2)

**目标**：
- 用 LSTM 建模交易序列 `(B, 32, F)`
- 在 **合成数据** 上达到 AUC-PR > 0.95
- 在 **Kaggle 序列** 上超过 Week 2 的 MLP baseline

**模型**：`Linear(F→64) → LSTM(hidden=128, layers=2, dropout=0.2) → 取最后时刻 → Linear(128→1)`

**产出**：两个数据集的指标对比表（train/val/test AUC-PR, AUC-ROC, Recall@FPR=0.001）

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 加载 Week 3 的序列张量（带 fallback）

若 `data/processed/*.pt` 不存在，本 cell 会**重建**：重跑合成生成器 + 序列切分的最小版本，保证 notebook 自包含可跑。

In [ ]:
import torch, numpy as np, pandas as pd
from pathlib import Path

proc_dir = PROJECT_ROOT / 'data' / 'processed'
need = ['synth_train.pt','synth_val.pt','synth_test.pt',
        'kaggle_train.pt','kaggle_val.pt','kaggle_test.pt']
missing = [f for f in need if not (proc_dir / f).exists()]
print('missing:', missing)

In [ ]:
# ── fallback: 只在 missing 时触发 ──
if missing:
    print('[fallback] rebuilding processed tensors from scratch...')
    proc_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)

    # ---- 合成数据（精简版：100 用户 × 20 天） ----
    N_U, N_D = 100, 20
    base = pd.Timestamp('2025-01-01')
    rows = []
    for u in range(N_U):
        mu = rng.choice([2.5, 3.5, 4.5])
        h_lat = rng.uniform(30, 50); h_lng = rng.uniform(-120, -70)
        pref = rng.dirichlet(np.ones(20) * 0.3)
        for d in range(N_D):
            for _ in range(rng.poisson(5)):
                ts = base + pd.Timedelta(days=int(d),
                                         hours=float(rng.uniform(0, 24)),
                                         minutes=int(rng.integers(0, 60)))
                rows.append((u, ts, float(np.exp(rng.normal(mu, 0.7))),
                             int(rng.choice(20, p=pref)),
                             float(h_lat + rng.normal(0, 0.05)),
                             float(h_lng + rng.normal(0, 0.05)),
                             int(rng.integers(0, 2)), 0))
    df = pd.DataFrame(rows, columns=['user_id','ts','amount','cat','lat','lng','dev','is_anomaly'])
    # 注入简化异常（金额飙升）
    idx = rng.choice(len(df), size=int(len(df) * 0.03), replace=False)
    df.loc[idx, 'amount'] *= rng.uniform(10, 30, size=len(idx))
    df.loc[idx, 'is_anomaly'] = 1
    df['ts_unix'] = df.ts.astype('int64') // 10**9
    df['hour_sin'] = np.sin(2*np.pi*df.ts.dt.hour/24)
    df['hour_cos'] = np.cos(2*np.pi*df.ts.dt.hour/24)
    df['amount_log'] = np.log1p(df['amount'])
    SF = ['amount_log','hour_sin','hour_cos','lat','lng','cat','dev']

    # Kaggle CSV
    csv = PROJECT_ROOT / 'data' / 'creditcard.csv'
    if not csv.exists():
        !pip install -q kaggle
        !kaggle datasets download -d mlg-ulb/creditcardfraud -p {PROJECT_ROOT / 'data'} --unzip
    dk = pd.read_csv(csv).sort_values('Time').reset_index(drop=True)
    dk['user_id'] = (((dk['Time']//60).astype('int64')*97 +
                     dk['Amount'].round().astype('int64')*53) % 500).astype('int32')
    dk['ts_unix'] = dk['Time'].astype('int64')
    dk['is_anomaly'] = dk['Class'].astype('int32')
    KF = [f'V{i}' for i in range(1,29)] + ['Amount']

    def split_time(d):
        q70, q85 = d.ts_unix.quantile(0.70), d.ts_unix.quantile(0.85)
        return (d[d.ts_unix<q70].copy(), d[(d.ts_unix>=q70)&(d.ts_unix<q85)].copy(),
                d[d.ts_unix>=q85].copy())
    def build(d, feats, L=32):
        xs, ys = [], []
        for _, g in d.sort_values(['user_id','ts_unix']).groupby('user_id', sort=False):
            a = g[feats].to_numpy('float32'); l = g['is_anomaly'].to_numpy('int64')
            if len(a) < L: continue
            for i in range(L, len(a)+1):
                xs.append(a[i-L:i]); ys.append(l[i-1])
        if not xs: return np.zeros((0,L,len(feats)),'float32'), np.zeros((0,),'int64')
        return np.stack(xs), np.array(ys,'int64')

    for name, dfi, feats in [('synth', df, SF), ('kaggle', dk, KF)]:
        tr, va, te = split_time(dfi)
        Xs = {}; Ys = {}
        for key, part in [('train', tr), ('val', va), ('test', te)]:
            X, y = build(part, feats)
            Xs[key], Ys[key] = X, y
        m = Xs['train'].reshape(-1, Xs['train'].shape[-1]).mean(0)
        s = Xs['train'].reshape(-1, Xs['train'].shape[-1]).std(0) + 1e-6
        for key in ('train','val','test'):
            Xs[key] = ((Xs[key] - m) / s).astype('float32')
            torch.save({'X': torch.from_numpy(Xs[key]),
                        'y': torch.from_numpy(Ys[key]).long()},
                       proc_dir / f'{name}_{key}.pt')
            print(f'rebuilt {name}_{key}.pt  X={Xs[key].shape}  pos={int(Ys[key].sum())}')
else:
    print('[fallback] skipped; all tensors present.')

## 2. 读入 Dataset / DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    def __init__(self, pt_path):
        d = torch.load(pt_path)
        self.X = d['X'].float()
        self.y = d['y'].float()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_loaders(prefix, batch=128):
    tr = SeqDataset(proc_dir / f'{prefix}_train.pt')
    va = SeqDataset(proc_dir / f'{prefix}_val.pt')
    te = SeqDataset(proc_dir / f'{prefix}_test.pt')
    print(f'{prefix}: train={len(tr)} val={len(va)} test={len(te)}  '
          f'F={tr.X.shape[-1]}  pos_tr={int(tr.y.sum())}')
    return (DataLoader(tr, batch, shuffle=True),
            DataLoader(va, batch, shuffle=False),
            DataLoader(te, batch, shuffle=False),
            tr.X.shape[-1])

tr_s, va_s, te_s, F_s = make_loaders('synth')
tr_k, va_k, te_k, F_k = make_loaders('kaggle')

## 3. LSTM 分类器

结构：
```
x : (B, L, F)
  │
  └─ Linear(F→64)           # 输入投影，让不同数据集用同一 hidden 结构
  └─ LSTM(input=64, hidden=128, num_layers=2, dropout=0.2, batch_first=True)
       output: (B, L, 128)
  └─ 取 last timestep: out[:, -1, :]
  └─ Dropout(0.3)
  └─ Linear(128→1)          # logit
```

**关于 `pack_padded_sequence`**：本任务所有窗口都是定长 L=32，不需要 pack；但保留注释片段展示变长序列的处理模板，Week 7+ 会真用到。

In [ ]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, in_dim, hidden=128, proj=64, num_layers=2, p_drop=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, proj)
        self.lstm = nn.LSTM(input_size=proj, hidden_size=hidden,
                            num_layers=num_layers, dropout=0.2, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(hidden, 1))

    def forward(self, x, lengths=None):
        # x: (B, L, F)
        z = torch.relu(self.proj(x))                    # (B, L, proj)

        # --- 变长序列模板（本任务定长，注释掉） ---
        # if lengths is not None:
        #     packed = nn.utils.rnn.pack_padded_sequence(
        #         z, lengths.cpu(), batch_first=True, enforce_sorted=False)
        #     out, (h, c) = self.lstm(packed)
        #     out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # else:
        out, (h, c) = self.lstm(z)                      # out: (B, L, H)
        last = out[:, -1, :]                            # 最后时刻 (B, H)
        return self.head(last).squeeze(-1)              # (B,)

# 快速 smoke
m = LSTMClassifier(F_s).to(device)
xb, yb = next(iter(tr_s))
print('forward shape:', m(xb.to(device)).shape, '  expected (B,) =', yb.shape)

## 4. 训练 + 评估工具（复用 Week 2 的评估器）

In [ ]:
import copy, time
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix

@torch.no_grad()
def predict(model, loader):
    model.eval()
    ss, ys = [], []
    for x, y in loader:
        x = x.to(device)
        ss.append(torch.sigmoid(model(x)).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ss), np.concatenate(ys)

def evaluate(model, loader):
    s, y = predict(model, loader)
    ap = average_precision_score(y, s) if y.sum() > 0 else float('nan')
    roc = roc_auc_score(y, s)           if y.sum() > 0 else float('nan')
    fpr, tpr, _ = roc_curve(y, s)
    idx = np.searchsorted(fpr, 0.001)
    rec = tpr[min(idx, len(tpr)-1)]
    return {'ap': ap, 'roc': roc, 'recall@fpr=0.001': rec, 'scores': s, 'labels': y}

def train_lstm(tr_loader, va_loader, in_dim, pos_weight,
               lr=1e-3, max_epochs=20, patience=4, tag=''):
    torch.manual_seed(SEED)
    model = LSTMClassifier(in_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    best_ap, best_state, bad = -1.0, None, 0
    for ep in range(1, max_epochs + 1):
        model.train(); t0, total = time.time(), 0.0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * x.size(0)
        val = evaluate(model, va_loader)
        print(f'[{tag}] ep{ep:02d}  loss={total/len(tr_loader.dataset):.4f}  '
              f"val_ap={val['ap']:.4f}  val_roc={val['roc']:.4f}  "
              f"rec@fpr=0.001={val['recall@fpr=0.001']:.3f}  ({time.time()-t0:.1f}s)")
        if val['ap'] > best_ap:
            best_ap = val['ap']; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f'[{tag}] early stop @ ep{ep}  best val_ap={best_ap:.4f}'); break
    model.load_state_dict(best_state)
    return model, best_ap

## 5. 训练：合成数据

预期：val AUC-PR > 0.95（异常注入明显，LSTM 应该轻松学到）。

In [ ]:
# pos_weight = #neg / #pos
tr_ds = tr_s.dataset
pw_s = float((tr_ds.y == 0).sum() / max((tr_ds.y == 1).sum().item(), 1))
print('synth pos_weight =', pw_s)

model_s, best_s = train_lstm(tr_s, va_s, in_dim=F_s, pos_weight=pw_s,
                             max_epochs=15, tag='synth')
test_s = evaluate(model_s, te_s)
print(f"\n[synth] TEST  AUC-PR={test_s['ap']:.4f}  AUC-ROC={test_s['roc']:.4f}  "
      f"Recall@FPR=0.001={test_s['recall@fpr=0.001']:.3f}")

## 6. 训练：Kaggle 序列

预期：val AUC-PR 高于 Week 2 MLP 的 0.70~0.75。如果不如 MLP，debug 方向：
- 伪 user_id 是否随机到噪声多？
- 序列标签只看最后一笔，可能漏掉窗口中间的欺诈信号 → 考虑 `max(y[i-L+1:i+1])` 作为窗口标签的对比。

In [ ]:
tr_ds_k = tr_k.dataset
pw_k = float((tr_ds_k.y == 0).sum() / max((tr_ds_k.y == 1).sum().item(), 1))
print('kaggle pos_weight =', pw_k)

model_k, best_k = train_lstm(tr_k, va_k, in_dim=F_k, pos_weight=pw_k,
                             max_epochs=15, tag='kaggle')
test_k = evaluate(model_k, te_k)
print(f"\n[kaggle] TEST  AUC-PR={test_k['ap']:.4f}  AUC-ROC={test_k['roc']:.4f}  "
      f"Recall@FPR=0.001={test_k['recall@fpr=0.001']:.3f}")

## 7. 对比表 + 混淆矩阵

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')

summary = pd.DataFrame([
    {'dataset':'synth',  'val_ap': best_s, 'test_ap': test_s['ap'],
     'test_roc': test_s['roc'], 'test_rec@fpr=0.001': test_s['recall@fpr=0.001']},
    {'dataset':'kaggle', 'val_ap': best_k, 'test_ap': test_k['ap'],
     'test_roc': test_k['roc'], 'test_rec@fpr=0.001': test_k['recall@fpr=0.001']},
])
print(summary.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, res) in zip(axes, [('synth', test_s), ('kaggle', test_k)]):
    y, s = res['labels'], res['scores']
    prec, rec, thr = precision_recall_curve(y, s)
    f1 = 2*prec*rec/(prec+rec+1e-9)
    bi = int(np.nanargmax(f1[:-1])); bt = thr[bi]
    yp = (s >= bt).astype(int)
    cm = confusion_matrix(y, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['pred0','pred1'], yticklabels=['true0','true1'])
    ax.set_title(f'{name}: cm @ maxF1 (thr={bt:.3f}, F1={f1[bi]:.3f})')
plt.tight_layout()

In [ ]:
# 保存 checkpoint
ckpt_dir = PROJECT_ROOT / 'checkpoints'
ckpt_dir.mkdir(exist_ok=True)
torch.save({'state_dict': model_s.state_dict(), 'in_dim': F_s}, ckpt_dir / 'lstm_synth.pt')
torch.save({'state_dict': model_k.state_dict(), 'in_dim': F_k}, ckpt_dir / 'lstm_kaggle.pt')
print('saved lstm checkpoints to', ckpt_dir)

## 8. 复盘笔记

1. **LSTM 4 个门**（画图自答）：
   - 遗忘门 f：控制上一时刻 cell state 保留多少
   - 输入门 i：控制新信息写入多少
   - 候选值 ĉ：当前候选 cell state
   - 输出门 o：控制 cell state 通过 tanh 输出到 hidden 的比例

2. **为什么 LSTM 缓解梯度消失？** cell state 是加法更新（`c_t = f·c_{t-1} + i·ĉ`），梯度沿加法通道近似恒等，比 RNN 的连乘稳定得多。

3. **合成 vs Kaggle 的 AUC-PR 差距反映了什么？**
   - 合成异常模式明显（金额×10~30），LSTM 轻松学到 → 反映模型上限。
   - Kaggle 是真实 PCA 特征 + 伪 user_id，难度接近上限 → 反映**问题本身**的困难。

4. **下周要进 Attention**：把 LSTM 的"用最后时刻"换成"对所有时刻加权"，看 AUC-PR 还能涨多少。